In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import random
import pickle
from copy import deepcopy
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F

SEED = 42
DATA_ROOT = "./data"
NUM_WORKERS = 0
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

SAVE_DIR = "/content/drive/MyDrive/ML_Project/project_files/GridSearch_Mahalanobis"
os.makedirs(SAVE_DIR, exist_ok=True)

GN_GROUPS = 32
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def torch_load_compat(path, map_location="cpu"):
    return torch.load(path, map_location=map_location, weights_only=False)

def get_best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_best_device()
PIN_MEM = (device.type == "cuda")
if device.type == "cuda":
    print(f"[INFO] Using device: {device}")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

MD_DIR = SAVE_DIR
os.makedirs(MD_DIR, exist_ok=True)

# existing files (adjust names if different on your drive)
MD_TASK1_STATS_PATH = os.path.join(MD_DIR, "best_epoch_mahalanobis_stats.pth")            # classes [0,1]
TASK2_BEST_STATS_PATH = os.path.join(MD_DIR, "task2_best_epoch_qda_stats_classes2,3.pth") # classes [2,3]
TOPK_PATH      = os.path.join(MD_DIR, "CS_class0-7_topk.pkl")
NEIGHBORS_PATH = os.path.join(MD_DIR, "CS_neighbors_class0-7.pkl")

# **NEW**: path to precomputed stats for classes 4,5 (you said these are already saved)
TASK3_STATS_45_PATH = os.path.join(MD_DIR, "task3_best_epoch_qda_stats_classes4,5.pth")

# prefer final fine-tuned weights produced by the earlier script (finetuned on 0..7)
FINAL_BEST_WEIGHTS = os.path.join(MD_DIR, "finetuned_task4_best.pth")
FINAL_BEST_WEIGHTS_SAVE = os.path.join(MD_DIR, "finetuned_task5_best.pth")

# we'll save stats for classes 8,9 here (renamed variable for clarity)
TASK5_BEST_STATS_PATH = os.path.join(MD_DIR, "task5_best_epoch_qda_stats_classes8,9.pth")  # will save 8,9 stats

# Also path where stats for 6,7 would be expected if previously computed
TASK4_STATS_67_PATH = os.path.join(MD_DIR, "task4_best_epoch_qda_stats_classes6,7.pth")  # expected for classes 6,7

# ---------------- ResNet / GroupNorm ----------------
def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)
        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                make_gn(planes)
            )
    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class ResNet18Backbone(nn.Module):
    def __init__(self, nf=64):
        super().__init__()
        self.nf = nf
        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)
        self.layer1 = nn.Sequential(BasicBlock(nf, nf), BasicBlock(nf, nf))
        self.layer2 = nn.Sequential(BasicBlock(nf, nf*2, 2), BasicBlock(nf*2, nf*2))
        self.layer3 = nn.Sequential(BasicBlock(nf*2, nf*4, 2), BasicBlock(nf*4, nf*4))
        self.layer4 = nn.Sequential(BasicBlock(nf*4, nf*8, 2), BasicBlock(nf*8, nf*8))
    def forward(self, x):
        x = torch.relu(self.gn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = F.avg_pool2d(x, x.shape[2])
        return x.view(x.size(0), -1)
    @property
    def out_dim(self):
        return self.nf * 8

class SingleHeadNet(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)
    def forward(self, x):
        return self.head(self.backbone(x))

# ---------------- Data utils ----------------
def get_cifar10_datasets():
    tf_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
    ])
    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
    ])
    train = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=tf_train)
    test  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tf_test)
    return train, test

def indices_for_classes(dataset, keep):
    t = dataset.targets if hasattr(dataset, "targets") else [dataset[i][1] for i in range(len(dataset))]
    return [i for i,y in enumerate(t) if int(y) in keep]

class RemapDataset(Dataset):
    def __init__(self, dataset, indices, keep_classes):
        self.dataset = dataset
        self.indices = indices
        self.keep_classes = sorted(keep_classes)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        x,y = self.dataset[self.indices[idx]]
        return x,y

def make_loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEM)

# ---------------- Fisher / EWC helpers (unchanged) ----------------
def load_fisher_data():
    with open(TOPK_PATH, "rb") as f: topk = pickle.load(f)
    with open(NEIGHBORS_PATH, "rb") as f: neigh = pickle.load(f)
    print(f"[INFO] Loaded Fisher data | TopK={len(topk)} | Neigh={len(neigh)}")
    return topk, neigh

def build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values):
    pm = dict(model.named_parameters())
    usable = [n for n in fisher_neighbors if n['name'] in pm]
    if not usable: return {}
    max_f = max((n['cs'] for n in usable), default=1.0) or 1.0
    buckets = {}
    for n in usable:
        buckets.setdefault(n['name'], []).append((int(n['index']), float(n['cs'])/max_f))
    ewc = {}
    for name, lst in buckets.items():
        lst.sort(key=lambda t:t[0])
        idxs = torch.tensor([i for i,_ in lst], device=device, dtype=torch.long)
        fish = torch.tensor([f for _,f in lst], device=device, dtype=torch.float32)
        flat = pm[name].view(-1)
        orig = torch.stack([neighbor_original_values[(name,int(i))] for i in idxs.tolist()]).to(flat.device, dtype=flat.dtype)
        ewc[name] = {'idxs': idxs, 'fish': fish, 'orig': orig}
    return ewc

def build_freeze_masks_and_cache(model, topk_list):
    masks, frozen_idxs, frozen_vals = {}, {}, {}
    pm = dict(model.named_parameters())
    by_name = {}
    for e in topk_list:
        n, i = e['name'], int(e['index'])
        if (n in pm) and (not n.startswith("head")):
            by_name.setdefault(n, []).append(i)
    for name, idxs in by_name.items():
        p = pm[name]
        flat = p.detach().view(-1)
        idxs_t = torch.tensor(idxs, device=flat.device, dtype=torch.long)
        m = torch.ones_like(p, dtype=torch.bool, device=p.device)
        mv = m.view(-1)
        mv[idxs_t] = False
        masks[name] = mv.view_as(p)
        frozen_idxs[name] = idxs_t
        frozen_vals[name] = flat.index_select(0, idxs_t).clone()
    return masks, frozen_idxs, frozen_vals

def apply_freeze_after_backward(model, masks):
    with torch.no_grad():
        for n,p in model.named_parameters():
            m = masks.get(n,None)
            if p.grad is not None and m is not None:
                p.grad.mul_(m.to(p.grad.dtype))

@torch.no_grad()
def apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals):
    for n,idxs in frozen_idxs.items():
        if n in param_map:
            flat = param_map[n].view(-1)
            flat.index_copy_(0, idxs, frozen_vals[n].to(flat.device, dtype=flat.dtype))

# ---------------- QDA helpers (compute stats returns no priors) ----------------
@torch.no_grad()
def compute_md_stats_from_loader(backbone: nn.Module, loader, class_ids: List[int]):
    """
    Returns: means, inv_covs, logdets
    (no priors) — compute per-class covariances and inverses.
    """
    backbone.eval()
    feats = []
    labels = []
    for x, y in loader:
        x = x.to(device)
        f = backbone(x).detach().cpu()
        feats.append(f)
        labels.append(y.detach().cpu())
    feats = torch.cat(feats, dim=0)    # [N,D]
    labels = torch.cat(labels, dim=0)  # [N]

    means, inv_covs, logdets = [], [], []

    for c in class_ids:
        cf = feats[labels == c]
        if cf.numel() == 0:
            raise RuntimeError(f"No samples found for class {c} to compute QDA stats.")

        mean = cf.mean(0)
        centered = cf - mean

        cov = torch.cov(centered.T)
        eps = 1e-5
        cov = cov + eps * torch.eye(cov.size(0))

        sign, logdet = torch.slogdet(cov)
        if sign.item() <= 0:
            cov = cov + 1e-3 * torch.eye(cov.size(0))
            sign, logdet = torch.slogdet(cov)

        inv = torch.inverse(cov)

        means.append(mean)
        inv_covs.append(inv)
        logdets.append(logdet)

    means = torch.stack(means, dim=0).to(device)       # [K,D]
    inv_covs = torch.stack(inv_covs, dim=0).to(device) # [K,D,D]
    logdets = torch.stack(logdets, dim=0).to(device)   # [K]
    return means, inv_covs, logdets

@torch.no_grad()
def qda_scores(backbone: nn.Module, x, means, inv_covs, logdets, priors):
    """
    Return g_k(x) = -0.5 * quad - 0.5 * logdet + log(pi_k)
    Output: [B, K] (bigger = more likely)
    """
    backbone.eval()
    f = backbone(x)  # [B, D]
    scores = []
    K = means.size(0)
    for k in range(K):
        diff = f - means[k]  # [B, D]
        quad = torch.sum((diff @ inv_covs[k]) * diff, dim=1)  # [B]
        gk = -0.5 * quad - 0.5 * logdets[k] + torch.log(priors[k] + 1e-20)
        scores.append(gk.unsqueeze(1))
    return torch.cat(scores, dim=1)  # [B, K]

# ---------------- Generic QDA evaluation helper (new) ----------------
@torch.no_grad()
def eval_loader_qda_nway(model, loader, qda_groups: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]]):
    """
    Generic evaluator that accepts a list of QDA groups.
    Each element in qda_groups is a tuple: (means, inv_covs, logdets, priors)
    It concatenates the per-group scores to produce a unified score over all classes.

    Returns: acc, confusion_matrix (labels inferred from number of classes)
    """
    model.eval()
    preds, labels = [], []
    # determine total number of classes from groups
    num_classes = sum([int(m.size(0)) for (m,_,_,_) in qda_groups])
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        all_scores = []
        for (means, invs, lds, priors) in qda_groups:
            scores = qda_scores(model.backbone, x, means.to(device), invs.to(device), lds.to(device), priors.to(device))  # [B, K_group]
            all_scores.append(scores)
        s_all = torch.cat(all_scores, dim=1)  # [B, num_classes]
        pred = torch.argmax(s_all, dim=1)
        preds.extend(pred.detach().cpu().numpy())
        labels.extend(y.detach().cpu().numpy())
    labels_np = np.array(labels)
    preds_np = np.array(preds)
    acc = 100.0 * np.mean(preds_np == labels_np)
    cm = confusion_matrix(labels_np, preds_np, labels=list(range(num_classes)))
    return acc, cm

# ---------------- load QDA stats helper (returns priors if saved) ----------------
def load_qda_stats_file(path, expected_classes):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Expected QDA stats file not found: {path}")
    st = torch_load_compat(path, map_location="cpu")
    classes = st.get("classes", expected_classes)
    assert list(classes) == list(expected_classes), f"Expected classes {expected_classes}, got {classes}"
    means = st["means"].to(device)
    inv   = st["inv_covs"].to(device)
    if "logdets" in st:
        logdet = st["logdets"].to(device)
    else:
        logdet_list = []
        for k in range(inv.size(0)):
            sign, ld = torch.slogdet(inv[k])
            logdet_list.append(-ld)
        logdet = torch.stack(logdet_list).to(device)
    priors = st.get("priors", None)
    if priors is None:
        priors = torch.ones(inv.size(0), device=device) / float(inv.size(0))
        try:
            print(f"[WARN] No 'priors' in {path} — falling back to uniform priors {priors.detach().cpu().numpy()}")
        except Exception:
            print(f"[WARN] No 'priors' in {path} — falling back to uniform priors")
    else:
        priors = priors.to(device)
        try:
            print(f"[INFO] Loaded priors from {path}: {priors.detach().cpu().numpy()}")
        except Exception:
            print(f"[INFO] Loaded priors from {path}")
    return means, inv, logdet, priors

# ---------------- Initialize expanded model -> 10 classes, copy rows 0..7 from fine-tuned 0..7 checkpoint ----------------
def init_expanded_model_task3_to10_copy0_5():
    """
    Load ONLY from FINAL_BEST_WEIGHTS (finetuned_task4_best.pth). No fallback.
    If Drive mount error 'Transport endpoint is not connected' occurs, attempt
    one forced remount and retry. If still fails or file absent/corrupt, raise
    RuntimeError with actionable instructions.
    """
    p = FINAL_BEST_WEIGHTS
    if not p:
        raise RuntimeError("FINAL_BEST_WEIGHTS path is empty. Set FINAL_BEST_WEIGHTS to the checkpoint path.")

    # quick existence check
    if not os.path.exists(p):
        raise RuntimeError(
            f"Required checkpoint not found at: {p}\n"
            "Put finetuned_task4_best.pth exactly at that path in your Google Drive."
        )

    ckpt = None
    loaded_path = None
    last_exc = None

    # try load once; if transport endpoint error, attempt remount once then retry
    def try_load(path):
        try:
            return torch_load_compat(path, map_location=device)
        except Exception as e:
            raise e

    try:
        ckpt = try_load(p)
        loaded_path = p
    except Exception as e:
        last_exc = e
        err_str = str(e)
        # specific handling for Colab drive transport error
        if "Transport endpoint is not connected" in err_str or "transport endpoint is not connected" in err_str:
            print(f"[WARN] Drive transport error when loading checkpoint: {err_str}")
            print("[INFO] Attempting one forced remount of Google Drive and retrying load...")
            try:
                from google.colab import drive as _drive
                _drive.mount('/content/drive', force_remount=True)
            except Exception as mount_exc:
                print(f"[WARN] Remount attempt failed: {mount_exc}")
            # retry load
            try:
                ckpt = try_load(p)
                loaded_path = p
            except Exception as e2:
                last_exc = e2
        else:
            # non-transport error (corrupt file, permission, etc.)
            pass

    if ckpt is None:
        raise RuntimeError(
            f"Could not load required checkpoint from: {p}\n\n"
            "Possible causes and remedies:\n"
            "  - Google Drive mount is broken. Re-mount in Colab:\n"
            "      from google.colab import drive\n"
            "      drive.mount('/content/drive', force_remount=True)\n"
            "  - The file is missing. Check listing:\n"
            f"      !ls -l {os.path.dirname(p)}\n"
            "  - The file is corrupted; replace it with a valid checkpoint.\n\n"
            f"Last load error: {repr(last_exc)}"
        )

    print(f"[INFO] Loaded checkpoint from: {loaded_path}")

    state4 = ckpt["state"] if isinstance(ckpt, dict) and "state" in ckpt else ckpt
    head_key = None
    for k in state4.keys():
        if k.endswith("head.weight") or ".head.weight" in k:
            head_key = k
            break

    src_head_rows = 0
    if head_key is not None:
        try:
            src_head_rows = int(state4[head_key].shape[0])
        except Exception:
            src_head_rows = 0
    else:
        print("[WARN] head.weight not found in checkpoint - will still try to load backbone if possible.")
        src_head_rows = 0

    src_num_classes = src_head_rows if src_head_rows and src_head_rows >= 1 else 4

    # build source model (num classes = src_num_classes) and load state
    backbone_src = ResNet18Backbone(64)
    model_src = SingleHeadNet(backbone_src, src_num_classes).to(device)
    try:
        model_src.load_state_dict(state4, strict=True)
        print(f"[INFO] Loaded checkpoint into model_src (num_classes={src_num_classes}) with strict=True")
    except Exception as e:
        model_src.load_state_dict(state4, strict=False)
        print(f"[WARN] strict load failed for model_src; partial load applied: {e}")

    # build destination 10-class model and copy backbone weights
    backbone10 = ResNet18Backbone(64)
    model10 = SingleHeadNet(backbone10, 10).to(device)
    try:
        model10.backbone.load_state_dict(model_src.backbone.state_dict(), strict=True)
    except Exception:
        model10.backbone.load_state_dict(model_src.backbone.state_dict(), strict=False)

    # Attempt to copy up to 8 head rows (0..7) from the source head to the new head
    want_copy = 8  # copy rows 0..7
    src_rows = model_src.head.weight.size(0) if hasattr(model_src.head, "weight") else 0
    dst_rows = model10.head.weight.size(0)
    copy_rows = min(src_rows, dst_rows, want_copy)
    with torch.no_grad():
        if copy_rows > 0:
            model10.head.weight[:copy_rows, :].copy_(model_src.head.weight[:copy_rows, :])
            model10.head.bias[:copy_rows].copy_(model_src.head.bias[:copy_rows])
        if copy_rows < want_copy:
            print(f"[WARN] requested copy up to 0..{want_copy-1} but only {src_rows} rows available in source; copied {copy_rows} rows.")
        else:
            print(f"[INFO] copied rows 0..{copy_rows-1} from source head into new 10-class head.")

    return model10

# ---------------- collect cosine helper (unchanged) ----------------
def collect_cosine_mean(model, W, loader):
    model.eval()
    device_loc = next(model.parameters()).device
    cos_accum = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device_loc)
            y = y.to(device_loc)
            feats = model.backbone(x)             # [B, D]
            w = W[y]                              # [B, D]
            cosine = F.cosine_similarity(feats, w, dim=1)  # [B]
            cos_accum.append(cosine.cpu())
    if len(cos_accum) == 0:
        return float('nan')
    return torch.cat(cos_accum).mean().item()
@torch.no_grad()
def normalize_weights_l2(model, eps=1e-12):
    for p in model.parameters():
        if not p.requires_grad:
            continue

        if p.ndim >= 2:
            w = p.view(p.size(0), -1)
            norms = w.norm(2, dim=1, keepdim=True).clamp_min(eps)
            w.div_(norms)
        else:
            # Usually skip bias
            pass

# ---------------- train on classes 8 & 9, use saved stats for 0..7 and 6,7 if available ----------------
def train_one_config_task3(lr_head, lr_backbone, bs, epochs, lambda_ewc, topk_fisher, fisher_neighbors):
    model = init_expanded_model_task3_to10_copy0_5()
    n_old = 8  # freeze classes 0,1,2,3

    # get references
    W = model.head.weight   # shape [6, feat_dim]
    B = model.head.bias     # shape [6]

    # build masks (float on params)
    weight_mask = torch.ones_like(W, device=W.device)
    bias_mask   = torch.ones_like(B, device=B.device)

    if n_old > 0:
        weight_mask[:n_old, :] = 0.0
        bias_mask[:n_old]      = 0.0

    # keep original values for safety restore after optimizer.step()
    head_orig = {
        "weight": W.data[:n_old, :].clone(),
        "bias":   B.data[:n_old].clone()
    }

    # hook functions (zero gradients on frozen rows)
    def _freeze_weight_grad(grad):
        return grad * weight_mask.to(grad.device)

    def _freeze_bias_grad(grad):
        return grad * bias_mask.to(grad.device)

    # register hooks once (model is local to this call)
    W.register_hook(_freeze_weight_grad)
    B.register_hook(_freeze_bias_grad)
    param_map = {n:p for n,p in model.named_parameters()}
    masks, frozen_idxs, frozen_vals = build_freeze_masks_and_cache(model, topk_fisher)

    # load precomputed stats for (0,1), (2,3) and (4,5) from saved files (must exist)
    try:
        m01, inv01, ld01, p01 = load_qda_stats_file(MD_TASK1_STATS_PATH, [0,1])
    except Exception as e:
        raise RuntimeError(f"Failed to load stats for classes [0,1]: {e}")
    try:
        m23, inv23, ld23, p23 = load_qda_stats_file(TASK2_BEST_STATS_PATH, [2,3])
    except Exception as e:
        raise RuntimeError(f"Failed to load stats for classes [2,3]: {e}")
    try:
        m45, inv45, ld45, p45 = load_qda_stats_file(TASK3_STATS_45_PATH, [4,5])
    except Exception as e:
        raise RuntimeError(f"Failed to load stats for classes [4,5]: {e}")

    # Try to load stats for classes 6,7 — this file may or may not exist.
    # For correct unified 10-way evaluation we require stats for 6,7 as well.
    try:
        m67, inv67, ld67, p67 = load_qda_stats_file(TASK4_STATS_67_PATH, [6,7])
        have_m67 = True
    except Exception:
        have_m67 = False

    # If m67 not available, cannot perform correct evaluation over 0..9
    if not have_m67:
        raise RuntimeError(f"Missing QDA stats for classes [6,7]. Please compute/save them at: {TASK4_STATS_67_PATH} before running this script for unified 10-way evaluation.")

    with torch.no_grad():
        cpu_cache = {n:p.view(-1).detach().cpu() for n,p in model.named_parameters()}
    neighbor_original_values = {}
    for n in fisher_neighbors:
        name, idx = n['name'], int(n['index'])
        if name in cpu_cache and idx < cpu_cache[name].numel():
            neighbor_original_values[(name,idx)] = cpu_cache[name][idx]
    ewc_tensors = build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values)

    optimizer = optim.SGD([
        {"params": model.backbone.parameters(), "lr": lr_backbone, "weight_decay":WEIGHT_DECAY},
        {"params": model.head.parameters(), "lr": lr_head, "weight_decay":0.0}
    ], momentum=MOMENTUM)

    train_set, test_set = get_cifar10_datasets()

    # training set: classes 8,9
    train_89 = RemapDataset(train_set, indices_for_classes(train_set,[8,9]), [8,9])

    # test splits (unchanged: test on 0..7 groups). We also build test_all as 0..9
    test_01 = RemapDataset(test_set, indices_for_classes(test_set,[0,1]), [0,1])
    test_23 = RemapDataset(test_set, indices_for_classes(test_set,[2,3]), [2,3])
    test_45 = RemapDataset(test_set, indices_for_classes(test_set,[4,5]), [4,5])
    test_67 = RemapDataset(test_set, indices_for_classes(test_set,[6,7]), [6,7])
    test_89 = RemapDataset(test_set, indices_for_classes(test_set,[8,9]), [8,9])
    test_all = RemapDataset(test_set, indices_for_classes(test_set,list(range(10))), list(range(10)))

    train_loader = make_loader(train_89, bs, True)
    train_89_eval_loader = make_loader(train_89, 256, False)

    test_01_loader = make_loader(test_01, 256, False)
    test_23_loader = make_loader(test_23, 256, False)
    test_45_loader = make_loader(test_45, 256, False)
    test_67_loader = make_loader(test_67, 256, False)
    test_89_loader = make_loader(test_89, 256, False)
    test_all_loader = make_loader(test_all, 256, False)

    best_avg, best_epoch = -1, 0
    best_state, best_results = None, None
    best_m89 = best_inv89 = best_ld89 = best_p89 = None

    for e in range(1, epochs+1):
        model.train(); loss_sum=0; correct=0; total=0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)  # labels are 8 or 9
            optimizer.zero_grad(set_to_none=True)

            logits = model(imgs)  # [B,10]

            # EWC penalty
            ewc_penalty = 0.0
            if lambda_ewc != 0 and len(ewc_tensors) > 0:
                for name, pack in ewc_tensors.items():
                    p = param_map[name].view(-1)
                    diff = p.index_select(0, pack['idxs']) - pack['orig']
                    ewc_penalty += (pack['fish'] * (diff**2)).sum()

            # full logits used for CrossEntropy; labels are raw (8 or 9)
            loss_ce = F.cross_entropy(logits, labels)
            loss = loss_ce + (lambda_ewc/2.0) * ewc_penalty
            loss.backward()

            apply_freeze_after_backward(model, masks)
            optimizer.step()
            normalize_weights_l2(model)
            with torch.no_grad():
                if n_old > 0:
                    W.data[:n_old, :] = head_orig["weight"]
                    B.data[:n_old]     = head_orig["bias"]
            param_map_after = {n:p.data for n,p in model.named_parameters()}
            apply_strict_freeze_after_step(param_map_after, frozen_idxs, frozen_vals)

            # compute training accuracy on (8,9) using logits[:,8:10]
            logits_new = logits[:, 8:10]
            labels_remap = labels - 8    # map to {0,1} for accuracy calc
            preds_task = logits_new.argmax(1)
            correct += (preds_task == labels_remap).sum().item()
            total += labels_remap.size(0)
            loss_sum += float(loss.detach().cpu())

        acc_train = 100.0 * correct / max(1,total)

        # compute QDA stats for (8,9) from current backbone (now returns means, invs, logdets)
        m89, inv89, ld89 = compute_md_stats_from_loader(model.backbone, train_89_eval_loader, class_ids=[8,9])

        count8 = 0
        count9 = 0
        for _, y_batch in train_89_eval_loader:
            ys = y_batch.detach().cpu().numpy()
            count8 += int((ys == 8).sum())
            count9 += int((ys == 9).sum())

        total_89 = float(count8 + count9)
        if total_89 <= 0:
            p89 = torch.tensor([0.5, 0.5], device=device)
        else:
            p89 = torch.tensor([count8 / total_89, count9 / total_89], device=device)
        # -------------------------------------------------------------------------

        # prepare groups in order: (0,1), (2,3), (4,5), (6,7), (8,9)
        groups = [
            (m01, inv01, ld01, p01),
            (m23, inv23, ld23, p23),
            (m45, inv45, ld45, p45),
            (m67, inv67, ld67, p67),
            (m89, inv89, ld89, p89)
        ]

        # ---------- evaluate each test loader against FULL unified logits ----------
        acc01, _   = eval_loader_qda_nway(model, test_01_loader, groups)
        acc23, _   = eval_loader_qda_nway(model, test_23_loader, groups)
        acc45, _   = eval_loader_qda_nway(model, test_45_loader, groups)
        acc67, _   = eval_loader_qda_nway(model, test_67_loader, groups)
        acc89, _   = eval_loader_qda_nway(model, test_89_loader, groups)
        acc_all, cm_all = eval_loader_qda_nway(model, test_all_loader, groups)
        # -------------------------------------------------------------------------------

        avg_acc = (acc01 + acc23 + acc45 + acc67 + acc89) / 5.0

        print(f"Epoch {e:03d} | Loss={loss_sum/len(train_loader):.4f} | Train(8,9)={acc_train:.2f}% | "
              f"01(QDA)={acc01:.2f}% | 23(QDA)={acc23:.2f}% | 45(QDA)={acc45:.2f}% | 67(QDA)={acc67:.2f}% | 89(QDA)={acc89:.2f}% | Unified10(QDA)={acc_all:.2f}% | Avg={avg_acc:.2f}%")

        if avg_acc > best_avg:
            best_avg = avg_acc
            best_epoch = e
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            best_results = (acc01, acc23, acc45, acc67, acc89, avg_acc)
            best_m89 = m89.detach().cpu()
            best_inv89 = inv89.detach().cpu()
            best_ld89 = ld89.detach().cpu()
            best_p89 = p89.detach().cpu()   # <-- save priors of the best epoch

    print(f"[INFO] Best Epoch: {best_epoch} | AvgAcc={best_results[5]:.2f}% | "
          f"01={best_results[0]:.2f}% | 23={best_results[1]:.2f}% | 45={best_results[2]:.2f}% | 67={best_results[3]:.2f}% | 89={best_results[4]:.2f}%")

    # save QDA stats for (8,9) including priors (best epoch)
    if best_m89 is None:
        raise RuntimeError("No best epoch found; training may have failed or epochs=0.")

    # ensure we have best_p89 (fallback to last computed p89 if needed)
    try:
        if best_p89 is None:
            # try to compute from last train_89_eval_loader
            count8 = 0
            count9 = 0
            for _, y_batch in train_89_eval_loader:
                ys = y_batch.detach().cpu().numpy()
                count8 += int((ys == 8).sum())
                count9 += int((ys == 9).sum())
            total_89 = float(count8 + count9)
            if total_89 <= 0:
                best_p89 = torch.tensor([0.5, 0.5], device="cpu")
            else:
                best_p89 = torch.tensor([count8 / total_89, count9 / total_89], device="cpu")
    except Exception:
        # last-resort fallback
        if best_p89 is None:
            best_p89 = torch.tensor([0.5,0.5], device="cpu")

    torch.save({
        "means": best_m89,
        "inv_covs": best_inv89,
        "logdets": best_ld89,
        "priors": best_p89,
        "classes": [8,9],
        "best_epoch": best_epoch,
        "avg_acc": best_avg,
        "results": {
            "acc_01": best_results[0],
            "acc_23": best_results[1],
            "acc_45": best_results[2],
            "acc_67": best_results[3],
            "acc_89": best_results[4],
            "avg": best_results[5]
        }
    }, TASK5_BEST_STATS_PATH)
    print(f"[INFO] Saved Task5 QDA stats (classes 8,9) -> {TASK5_BEST_STATS_PATH}")

    return best_avg, best_state

# ---------------- Grid search wrapper (unchanged except param lists) ----------------
def grid_search_finetune_task5():
    topk_fisher, fisher_neighbors = load_fisher_data()

    LR_HEADS = [0.01,0.1, 0.2, 0.3, 0.7]
    LR_BACKBONES = [1e-5]
    LAMBDAS = [2]
    BATCH_SIZES = [64,128,256]
    EPOCHS_LIST = [10]

    best_acc, best_weights, best_cfg = -1, None, None

    for lr_h in LR_HEADS:
        for lr_b in LR_BACKBONES:
            for lam in LAMBDAS:
                for bs in BATCH_SIZES:
                    for ep in EPOCHS_LIST:
                        print("\n" + "="*70)
                        print(f"🚀 Task5 Config | lr_head={lr_h} | lr_backbone={lr_b} | λ={lam} | bs={bs} | epochs={ep}")
                        print("="*70)

                        acc, weights = train_one_config_task3(
                            lr_head=lr_h, lr_backbone=lr_b,
                            bs=bs, epochs=ep, lambda_ewc=lam,
                            topk_fisher=topk_fisher, fisher_neighbors=fisher_neighbors
                        )

                        if acc > best_acc:
                            best_acc = acc
                            best_weights = deepcopy(weights)
                            best_cfg = {"lr_head":lr_h,"lr_backbone":lr_b,"lambda":lam,"batch_size":bs,"epochs":ep}

    print("\n" + "#"*70)
    print(f"[DONE] Task5 Best AvgAcc={best_acc:.2f}% | Config={best_cfg}")
    print("#"*70)

    torch.save(best_weights, FINAL_BEST_WEIGHTS_SAVE)
    print(f"[INFO] Saved Task5 best weights -> {FINAL_BEST_WEIGHTS_SAVE}")

if __name__ == "__main__":
    grid_search_finetune_task5()